# Parte 1 — Notebook 3: Clasificador de tapitas (CNN)
## De tus videos a un modelo entrenado

En este notebook construyes **tu propio dataset** de tapitas a partir de video y entrenas una **CNN desde cero** (la misma idea de los NB 1 y 2, ahora con *tus* datos). El modelo resultante se prueba en vivo en el NB 4 y te servirá para tu proyecto.

**Necesitas 4 videos** (uno por clase): `rojo`, `azul`, `amarillo` y `fondo` (la zona vacía). Puedes **grabarlos desde aquí** (si tu cámara está disponible) o **subir** los que ya grabaste.

> Consejo: graba videos **cortos** (20–40 s) moviendo la tapita dentro de la zona de inspección, con la iluminación real del laboratorio. Es mejor tener variedad que muchos frames idénticos.

In [ ]:
import os, glob, shutil, random, time, io
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from sklearn.metrics import confusion_matrix, classification_report
import ipywidgets as widgets
from IPython.display import display, clear_output

random.seed(42); np.random.seed(42); tf.random.set_seed(42)

# Clases del proyecto (en orden alfabetico, como las leera Keras)
CLASES = ['amarillo', 'azul', 'fondo', 'rojo']

# Rutas robustas: funciona si el cwd es la raiz del proyecto o la carpeta del notebook
DATA = 'data' if os.path.isdir('data') else '../data'
VIDEO_DIR = os.path.join(DATA, 'videos_tapitas')   # aqui van los 4 videos
TAPITAS_DIR = os.path.join(DATA, 'tapitas')         # aqui iran los frames por clase
SPLIT_DIR = os.path.join(DATA, 'tapitas_split')     # train/validation
os.makedirs(VIDEO_DIR, exist_ok=True)

# Deteccion de WSL (en WSL no hay camara: usa la opcion de SUBIR videos)
def es_wsl():
    try:
        with open('/proc/version') as f: return 'microsoft' in f.read().lower()
    except Exception: return False

print(f'TensorFlow {tf.__version__} — listo. Clases: {CLASES}')
if es_wsl():
    print('Entorno WSL detectado: la grabacion por camara no esta disponible aqui; usa la opcion de SUBIR videos.')

### Función de extracción de frames
Ejecuta esta celda (define la función que convierte cada video en imágenes, descartando frames borrosos).

In [ ]:
def video_to_frames(video_path, output_folder, clase, interval_seconds=0.5,
                    min_sharpness=30.0, detect_stability=False):
    """Extrae frames de un video, descartando los borrosos."""
    os.makedirs(output_folder, exist_ok=True)
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'  No se pudo abrir: {video_path}'); return 0
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    frame_interval = max(1, int(fps * interval_seconds))
    prev_gray = None; count = saved = skipped = 0
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret: break
        if count % frame_interval == 0:
            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            if cv2.Laplacian(gray, cv2.CV_64F).var() < min_sharpness:
                skipped += 1; count += 1; continue
            if detect_stability and prev_gray is not None:
                if np.mean(cv2.absdiff(gray, prev_gray)) > 15:
                    prev_gray = gray; count += 1; continue
            prev_gray = gray
            cv2.imwrite(os.path.join(output_folder, f'frame_{clase}_{saved:05d}.jpg'), frame)
            saved += 1
        count += 1
    cap.release()
    print(f'  {clase}: {saved} frames guardados ({skipped} borrosos descartados)')
    return saved

## Paso 1 — Consigue tus 4 videos

Elige **una** de las dos opciones (A o B). Cada video se guarda como `rojo.mp4`, `azul.mp4`, `amarillo.mp4`, `fondo.mp4`.

### Opción A — Grabar desde la cámara
Se abre una ventana por clase: mueve la tapita dentro de la zona y presiona **q** para terminar. (No disponible en WSL.)

In [ ]:
def grabar_video(clase, segundos=25, camera_index=0):
    if es_wsl():
        print('WSL: usa la opcion B (subir videos).'); return
    out_path = os.path.join(VIDEO_DIR, f'{clase}.mp4')
    cap = cv2.VideoCapture(camera_index)
    if not cap.isOpened():
        print(f'No se pudo abrir la camara (indice {camera_index}).'); return
    cap.set(cv2.CAP_PROP_FRAME_WIDTH, 640); cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)
    w = int(cap.get(3)) or 640; h = int(cap.get(4)) or 480
    writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'), 20.0, (w, h))
    t0 = time.time()
    print(f"Grabando '{clase}' ~{segundos}s. Mueve la tapita. 'q' para terminar antes.")
    while True:
        ret, frame = cap.read()
        if not ret: break
        writer.write(frame)
        rem = segundos - (time.time() - t0)
        disp = frame.copy()
        cv2.putText(disp, f'{clase}  {max(0, rem):.0f}s  (q=fin)', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow('Grabando (q=fin)', disp)
        if (cv2.waitKey(1) & 0xFF) == ord('q') or rem <= 0: break
    cap.release(); writer.release(); cv2.destroyAllWindows()
    print(f'Guardado: {out_path}')

_botones = []
_out_grab = widgets.Output()
for _c in CLASES:
    _b = widgets.Button(description=f'Grabar {_c}', button_style='info')
    def _mk(cl):
        def _h(_):
            with _out_grab:
                clear_output(wait=True); grabar_video(cl)
        return _h
    _b.on_click(_mk(_c)); _botones.append(_b)
display(widgets.HBox(_botones)); display(_out_grab)

### Opción B — Subir videos
Sube el video correspondiente a cada clase y luego presiona **Guardar videos**.

In [ ]:
_uploaders = {c: widgets.FileUpload(accept='video/*', multiple=False) for c in CLASES}
_filas = [widgets.HBox([widgets.Label(f'{c}:', layout=widgets.Layout(width='90px')), _uploaders[c]]) for c in CLASES]
_btn_guardar = widgets.Button(description='Guardar videos', button_style='success', icon='save')
_out_up = widgets.Output()

def _guardar(_):
    with _out_up:
        clear_output(wait=True)
        for c, up in _uploaders.items():
            if not up.value: continue
            item = list(up.value.values())[0] if isinstance(up.value, dict) else up.value[0]
            content = item['content']
            with open(os.path.join(VIDEO_DIR, f'{c}.mp4'), 'wb') as f:
                f.write(content)
            print(f'Guardado {c}.mp4 ({len(content)//1024} KB)')
        print('Listo. Continua con el Paso 2.')

_btn_guardar.on_click(_guardar)
display(widgets.VBox(_filas + [_btn_guardar, _out_up]))

## Paso 2 — Extraer los frames
Convierte los 4 videos en imágenes dentro de `data/tapitas/<clase>/`.

In [ ]:
INTERVALO_S = 0.5   # segundos entre frames (sube/baja para tener mas/menos imagenes)
NITIDEZ_MIN = 30.0  # descarta frames mas borrosos que esto

total = {}
for c in CLASES:
    vp = os.path.join(VIDEO_DIR, f'{c}.mp4')
    if os.path.exists(vp):
        dest = os.path.join(TAPITAS_DIR, c)
        if os.path.isdir(dest): shutil.rmtree(dest)
        total[c] = video_to_frames(vp, dest, c, INTERVALO_S, NITIDEZ_MIN)
    else:
        print(f'  (falta {c}.mp4 — grabalo o subelo en el Paso 1)')

# Resumen y vista previa
if total:
    print('\nImagenes por clase:', total)
    fig, axes = plt.subplots(1, len(total), figsize=(4*len(total), 4))
    if len(total) == 1: axes = [axes]
    for ax, c in zip(axes, total):
        imgs = glob.glob(os.path.join(TAPITAS_DIR, c, '*.jpg'))
        if imgs: ax.imshow(load_img(random.choice(imgs), target_size=(96, 96)))
        ax.set_title(f'{c} ({total[c]})'); ax.axis('off')
    plt.suptitle('Vista previa del dataset'); plt.tight_layout(); plt.show()

## Paso 3 — Dividir y entrenar la CNN
Se separa el dataset en *train*/*validation* y se entrena una CNN (misma arquitectura de los NB 1–2, adaptada al número de clases).

In [ ]:
# --- Split train/validation ---
def split_dataset(src, out, ratio=0.8):
    if os.path.isdir(out): shutil.rmtree(out)
    clases = sorted([d for d in os.listdir(src) if os.path.isdir(os.path.join(src, d))])
    random.seed(42)
    for c in clases:
        imgs = sorted(glob.glob(os.path.join(src, c, '*.jpg')))
        random.shuffle(imgs)
        n = int(len(imgs) * ratio)
        for sub, lst in [('train', imgs[:n]), ('validation', imgs[n:])]:
            d = os.path.join(out, sub, c); os.makedirs(d, exist_ok=True)
            for p in lst: shutil.copy(p, os.path.join(d, os.path.basename(p)))
    return clases

clases = split_dataset(TAPITAS_DIR, SPLIT_DIR, 0.8)
num_classes = len(clases)
print('Clases:', clases, '| num_classes =', num_classes)
for sub in ['train', 'validation']:
    n = sum(len(glob.glob(os.path.join(SPLIT_DIR, sub, c, '*.jpg'))) for c in clases)
    print(f'  {sub}: {n} imagenes')

In [ ]:
IMG_SIZE = 96

train_datagen = ImageDataGenerator(
    rescale=1./255, rotation_range=20, width_shift_range=0.1, height_shift_range=0.1,
    brightness_range=[0.8, 1.2], zoom_range=0.15, fill_mode='nearest')
val_datagen = ImageDataGenerator(rescale=1./255)

train_set = train_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'train'), target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16, class_mode='categorical')
val_set = val_datagen.flow_from_directory(
    os.path.join(SPLIT_DIR, 'validation'), target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=16, class_mode='categorical', shuffle=False)
print('Indices de clase:', train_set.class_indices)

In [ ]:
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
    MaxPooling2D(2),
    Conv2D(32, (3, 3), activation='relu'),
    MaxPooling2D(2),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D(2),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
EPOCHS = 30
history = model.fit(train_set, epochs=EPOCHS, validation_data=val_set)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))
r = range(1, len(history.history['accuracy'])+1)
a1.plot(r, history.history['accuracy'], 'b-', label='Train')
a1.plot(r, history.history['val_accuracy'], 'r-', label='Val')
a1.set_title('Accuracy'); a1.set_xlabel('Epoca'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(r, history.history['loss'], 'b-', label='Train')
a2.plot(r, history.history['val_loss'], 'r-', label='Val')
a2.set_title('Loss'); a2.set_xlabel('Epoca'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
val_set.reset()
preds = model.predict(val_set, verbose=0)
y_pred = np.argmax(preds, axis=1); y_true = val_set.classes
labels = list(val_set.class_indices.keys())
print(classification_report(y_true, y_pred, target_names=labels, zero_division=0))
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.xlabel('Prediccion'); plt.ylabel('Real'); plt.title('Matriz de confusion'); plt.tight_layout(); plt.show()

## Paso 4 — Exportar el modelo
Guarda el modelo y su metadata. El **NB 4** lo cargará para probarlo en vivo.

In [ ]:
import json as _json
MODELOS_DIR = os.path.join(DATA, os.pardir, 'modelos')  # <raiz>/modelos (compartido con el NB de prueba)
os.makedirs(MODELOS_DIR, exist_ok=True)
MODEL_OUT = os.path.join(MODELOS_DIR, 'tapitas_cnn.h5')
model.save(MODEL_OUT)
meta = {
    'class_names': labels, 'img_size': IMG_SIZE, 'preprocessing': 'rescale',
    'model_type': 'cnn', 'dataset_dir': TAPITAS_DIR,
    'val_accuracy': round(float(history.history['val_accuracy'][-1]), 4),
}
with open(MODEL_OUT.rsplit('.', 1)[0] + '.json', 'w') as f:
    _json.dump(meta, f, indent=2, ensure_ascii=False)
print('Modelo guardado en:', os.path.abspath(MODEL_OUT))
print('Val accuracy:', meta['val_accuracy'])
print('\nAhora abre el NB 4 (probar_modelo) para probarlo en vivo.')